<a href="https://colab.research.google.com/github/luladc/IA2/blob/main/LAB3IA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Montaje, Carga, Limpieza y Secuencias**

En esta primera celda cargamos los datos del sensor T1. El dataset tiene 52,560 registros (un año de observaciones cada 10 minutos). Interpolamos valores nulos para no perder el hilo temporal y aplicamos Data Augmentation (inyectando un 1% de ruido gaussiano) al conjunto de entrenamiento para que la red no memorice la serie exacta. Finalmente, creamos secuencias para mirar 6 horas al pasado y predecir 1 hora hacia el futuro.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive
import os
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math

# ==========================================
# 1. PREPARACIÓN Y LIMPIEZA DE DATOS
# ==========================================
drive.mount('/content/drive')
ruta_csv = '/content/drive/MyDrive/DATASETS/solar_10_minutes_dataset.csv'

# Usamos 'timestamp' como índice de tiempo
df = pd.read_csv(ruta_csv, parse_dates=['timestamp'], index_col='timestamp')

# Elegimos la estación 'T1'
columna_objetivo = 'T1'

# Rellenar vacíos
df[columna_objetivo] = df[columna_objetivo].interpolate(method='linear')

# Escalado a [0, 1]
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[[columna_objetivo]].values)

# Ventanas Temporales: Mirar 36 pasos (6h) para predecir a 6 pasos (1h futuro)
def create_sequences_horizon(data, seq_length, horizon):
    xs, ys = [], []
    for i in range(len(data) - seq_length - horizon + 1):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length + horizon - 1])
    return np.array(xs), np.array(ys)

SEQ_LEN = 36
HORIZONTE = 6
X, y = create_sequences_horizon(data_scaled, SEQ_LEN, HORIZONTE)

# División Cronológica
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

X_train = torch.tensor(X[:train_size], dtype=torch.float32)
y_train = torch.tensor(y[:train_size], dtype=torch.float32)
X_val = torch.tensor(X[train_size:train_size+val_size], dtype=torch.float32)
y_val = torch.tensor(y[train_size:train_size+val_size], dtype=torch.float32)
X_test = torch.tensor(X[train_size+val_size:], dtype=torch.float32)
y_test = torch.tensor(y[train_size+val_size:], dtype=torch.float32)

# REGULARIZACIÓN: Data Augmentation
ruido = torch.randn_like(X_train) * 0.01
X_train_aug = X_train + ruido

# OPTIMIZACIÓN: Mini-batch
train_loader = DataLoader(TensorDataset(X_train_aug, y_train), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

print(f"Total secuencias de entrenamiento: {len(X_train)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total secuencias de entrenamiento: 36763


# **Arquitectura de la Red Neuronal (RNN, LSTM, GRU)**
Definimos una clase que encapsula las tres arquitecturas para poder intercambiarlas fácilmente. Incluimos explícitamente Batch Normalization y Dropout para evitar el sobreajuste.

In [ ]:
# ==========================================
# 2. ARQUITECTURA DE LA RED NEURONAL (Limpia)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class ModeloSolar(nn.Module):
    def __init__(self, model_type='LSTM', input_size=1, hidden_size=64, num_layers=2):
        super(ModeloSolar, self).__init__()

        # Reducimos el dropout interno a 10%
        if model_type == 'RNN':
            self.recurrent = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=0.1)
        elif model_type == 'GRU':
            self.recurrent = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=0.1)
        else:
            self.recurrent = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.1)

        # Quitamos el BatchNorm y el Dropout externo
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.recurrent(x)
        if isinstance(out, tuple):
            out = out[0]

        out = out[:, -1, :]
        return self.fc(out)

# **Motor de Entrenamiento Optimizado con Checkpoints**
Implementamos el optimizador Adam y el sistema que guarda el estado de la red para prevenir la pérdida de progreso ante interrupciones. También se implementa el Early Stopping que detendrá el entrenamiento si el modelo no mejora en el conjunto de validación tras varias épocas.

In [ ]:
import os

# ==========================================
# 3. MOTOR DE ENTRENAMIENTO LIMPIO
# ==========================================
def entrenar_modelo_limpio(model_type, epochs=40, patience=10):
    print(f"\n[{model_type}] Entrenando sin restricciones extremas...")
    modelo = ModeloSolar(model_type=model_type).to(device)

    # Adam clásico sin weight_decay
    optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    early_stop_counter = 0
    best_model = None

    for epoch in range(epochs):
        modelo.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = modelo(X_batch)
            loss = criterion(y_pred, y_batch) # ¡Corregido! Dimensiones alineadas
            loss.backward()
            optimizer.step()

        modelo.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = modelo(X_batch)
                val_loss += criterion(y_pred, y_batch).item()

        avg_val_loss = val_loss / len(val_loader)

        # Imprimir progreso
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"[{model_type}] Epoch {epoch+1}/{epochs} | Val Loss: {avg_val_loss:.6f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # Guardamos en memoria RAM los mejores pesos
            best_model = modelo.state_dict()
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print(f"[{model_type}] Early Stopping en la época {epoch+1}.")
                break

    if best_model is not None:
        modelo.load_state_dict(best_model)
    return modelo

# Ejecuta los tres modelos
modelo_rnn = entrenar_modelo_limpio('RNN')
modelo_lstm = entrenar_modelo_limpio('LSTM')
modelo_gru = entrenar_modelo_limpio('GRU')


[RNN] Entrenando sin restricciones extremas...
[RNN] Epoch 1/40 | Val Loss: 0.017680
[RNN] Epoch 5/40 | Val Loss: 0.015781
[RNN] Epoch 10/40 | Val Loss: 0.016199
[RNN] Epoch 15/40 | Val Loss: 0.014298
[RNN] Epoch 20/40 | Val Loss: 0.013701
[RNN] Epoch 25/40 | Val Loss: 0.016539
[RNN] Epoch 30/40 | Val Loss: 0.013283
[RNN] Epoch 35/40 | Val Loss: 0.012916
[RNN] Epoch 40/40 | Val Loss: 0.013555

[LSTM] Entrenando sin restricciones extremas...
[LSTM] Epoch 1/40 | Val Loss: 0.017134
[LSTM] Epoch 5/40 | Val Loss: 0.014199
[LSTM] Epoch 10/40 | Val Loss: 0.013866
[LSTM] Epoch 15/40 | Val Loss: 0.013154
[LSTM] Epoch 20/40 | Val Loss: 0.011866
[LSTM] Epoch 25/40 | Val Loss: 0.012181
[LSTM] Early Stopping en la época 29.

[GRU] Entrenando sin restricciones extremas...
[GRU] Epoch 1/40 | Val Loss: 0.016575
[GRU] Epoch 5/40 | Val Loss: 0.014129
[GRU] Epoch 10/40 | Val Loss: 0.013380
[GRU] Epoch 15/40 | Val Loss: 0.012157
[GRU] Epoch 20/40 | Val Loss: 0.012311
[GRU] Epoch 25/40 | Val Loss: 0.01266

# **Evaluación Cuantitativa (Métricas de Regresión) y Gráfico Interactivo**
En esta celda aplicamos métricas de regresión (MSE, RMSE, MAE y R2 Score) para comparar cuantitativamente la precisión de las tres arquitecturas, y generamos un gráfico interactivo con Plotly para comparar visualmente sus predicciones con el conjunto de prueba.

In [ ]:
# ==========================================
# 4. EVALUACIÓN Y MÉTRICAS
# ==========================================

# 1. Poner modelos en modo evaluación
modelo_rnn.eval()
modelo_lstm.eval()
modelo_gru.eval()

# 2. Hacer predicciones en el set de PRUEBA
with torch.no_grad():
    X_test_gpu = X_test.to(device)
    pred_rnn = modelo_rnn(X_test_gpu).cpu().numpy()
    pred_lstm = modelo_lstm(X_test_gpu).cpu().numpy()
    pred_gru = modelo_gru(X_test_gpu).cpu().numpy()
    y_real = y_test.numpy()

# 3. Revertir la normalización a la escala real de Radiación/Potencia
pred_rnn_real = scaler.inverse_transform(pred_rnn)
pred_lstm_real = scaler.inverse_transform(pred_lstm)
pred_gru_real = scaler.inverse_transform(pred_gru)
y_test_real = scaler.inverse_transform(y_real.reshape(-1, 1))

# 4. Cálculo de Métricas Matemáticas de Regresión
def calcular_metricas(y_true, y_pred, nombre_modelo):
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {nombre_modelo} ---")
    print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.2f}")
    print(f"MAE (Error Absoluto Medio): {mae:.2f}")
    print(f"R2 Score (Precisión de tendencia): {r2:.4f}\n")

print("MÉTRICAS DE EVALUACIÓN (PREDICCIÓN A 1 HORA FUTURA):")
calcular_metricas(y_test_real, pred_rnn_real, "RNN Clásica")
calcular_metricas(y_test_real, pred_lstm_real, "LSTM")
calcular_metricas(y_test_real, pred_gru_real, "GRU")

# 5. VISUALIZACIÓN INTERACTIVA (Primeros 400 pasos)
PASOS_A_MOSTRAR = 400

fig = go.Figure()

# Línea de Datos Reales
fig.add_trace(go.Scatter(y=y_test_real[:PASOS_A_MOSTRAR].flatten(),
                         mode='lines', name='Valor Real', line=dict(color='black', width=2)))

# Línea LSTM
fig.add_trace(go.Scatter(y=pred_lstm_real[:PASOS_A_MOSTRAR].flatten(),
                         mode='lines', name='Predicción LSTM', line=dict(color='blue', dash='dot')))

# Línea GRU
fig.add_trace(go.Scatter(y=pred_gru_real[:PASOS_A_MOSTRAR].flatten(),
                         mode='lines', name='Predicción GRU', line=dict(color='red', dash='dash')))

fig.update_layout(title='Predicción a 1 Hora Futura: Modelos Recurrentes vs Realidad',
                  xaxis_title='Intervalos Temporales (10 min c/u)',
                  yaxis_title='Radiación / Potencia (T1)',
                  hovermode="x unified",
                  template='plotly_white')
# Línea RNN Clásica
fig.add_trace(go.Scatter(y=pred_rnn_real[:PASOS_A_MOSTRAR].flatten(),
                         mode='lines', name='Predicción RNN', line=dict(color='orange', dash='dash')))

fig.show()

MÉTRICAS DE EVALUACIÓN (PREDICCIÓN A 1 HORA FUTURA):
--- RNN Clásica ---
RMSE (Raíz del Error Cuadrático Medio): 6.79
MAE (Error Absoluto Medio): 4.98
R2 Score (Precisión de tendencia): 0.7701

--- LSTM ---
RMSE (Raíz del Error Cuadrático Medio): 6.11
MAE (Error Absoluto Medio): 3.68
R2 Score (Precisión de tendencia): 0.8142

--- GRU ---
RMSE (Raíz del Error Cuadrático Medio): 6.08
MAE (Error Absoluto Medio): 3.64
R2 Score (Precisión de tendencia): 0.8155

